# Lab 5 — Pandas: Making Broken Data Trustworthy
**ARTI 303 — Programming for AI**

**Name:** _Reema Al Shawyah_  **Student ID:** _2250002150_  **Section:** _AI01_

---

### What this lab is really about

Most of an AI engineer's time is not spent on models. It's spent
getting data into a state where a model can be trusted with it.

That work is unglamorous and it is where the expensive mistakes happen —
because bad data rarely *crashes* anything. It produces numbers. Wrong
ones, confidently, with no error message.

So this lab isn't a tour of Pandas features. You get one deliberately
broken dataset, and you make it model-ready. Each tool appears because
a problem demanded it.

---

## Setup

Your course folder, with one environment shared by every lab:

```
arti303_lab/
├── .venv/
├── requirements.txt
├── lab01.ipynb
├── lab02.ipynb
├── lab03.ipynb
├── lab04.ipynb
├── lab05.ipynb        <- this lab
├── data/
└── ...
```

**1.** Open `arti303_lab` in VS Code (`File → Open Folder`) — the folder
itself, not a lab file inside it.

**2.** Activate the environment:

```bash
source .venv/bin/activate          # macOS / Linux / Git Bash
.venv\Scripts\Activate.ps1         # Windows PowerShell
```

Your prompt must start with `(.venv)`. If it shows `(base)`, you're in
Anaconda's environment and packages will install to the wrong place.

> No `.venv` yet? `python -m venv .venv` from this folder, then activate.

**3.** Install Pandas and record it:

```bash
python -m pip install pandas
python -m pip freeze > requirements.txt
```

**4.** Point this notebook at the same environment:
`Cmd+Shift+P` (macOS) / `Ctrl+Shift+P` (Windows) → **Python: Select
Interpreter** → the entry showing `.venv`. Then restart the kernel.

### Check your setup

In [1]:
import os
import sys

problems = []

if sys.prefix != sys.base_prefix:
    print("[ok]   Running inside a virtual environment")
else:
    print("[!]    NOT in a virtual environment")
    problems.append("Select the .venv interpreter, then restart the kernel")
print("       using:", sys.executable)
print()

try:
    import pandas as pd
    import numpy as np
    print("[ok]   pandas", pd.__version__, "| numpy", np.__version__)
except ImportError:
    print("[!]    pandas is not installed in this environment")
    problems.append("Run: python -m pip install pandas")

if os.path.exists("requirements.txt"):
    lines = [l for l in open("requirements.txt").read().splitlines() if l.strip()]
    if any("pandas" in l.lower() for l in lines):
        print("[ok]   pandas is recorded in requirements.txt")
    else:
        print("[!]    pandas is missing from requirements.txt")
        problems.append("Run: python -m pip freeze > requirements.txt")
    if len(lines) > 100:
        print("[!]    requirements.txt has", len(lines), "packages - that looks like Anaconda's base")
        problems.append("Delete it, activate .venv, freeze again")
else:
    print("[!]    No requirements.txt found")
    problems.append("Run: python -m pip freeze > requirements.txt")

print()
print("Fix these first:" if problems else "Setup looks good.")
for item in problems:
    print("  -", item)

[ok]   Running inside a virtual environment
       using: c:\Users\rimaa\Downloads\Telegram Desktop\lab05_solved\.venv\Scripts\python.exe

[ok]   pandas 3.0.6 | numpy 2.5.3
[ok]   pandas is recorded in requirements.txt

Setup looks good.


---
# Part 1 · From twelve lines to one

In an earlier lab you wrote a function to load student records: open the
file, skip the header, split each line, convert the types, catch the
rows that failed. Around twelve lines, and it only handled one file
shape.

Here is the same job in Pandas.

First, let's create the data files we'll use throughout.

In [2]:
import os
os.makedirs("data", exist_ok=True)

students_raw = """student_id,name,gender,gpa,credits,dept,status
1001,Sara Alotaibi,Female,4.60,95,CE,active
1002,  faisal alharbi ,male,3.90,88,ce,active
1003,Lama Alqahtani,F,4.85,102,CE,active
1004,Omar Alshehri,M,pending,76,CS,active
1005,Noura Alzahrani,female,2.95,64,cs,inactive
1006,Khalid Almutairi,Male,3.40,,CE,active
1007,Reem Alharbi,F,4.50,91,BME,active
1003,Lama Alqahtani,F,4.85,102,CE,active
1008,Yousef Alqahtani,M,1.85,45,CS,inactive
1009,Hala Alotaibi, female,4.20,83,BME,active
1010,Bader Alanazi,M,3.75,79,CE,active
1011,Maha Alsubaie,F,--,58,BME,inactive
1012,Turki Aldossari,male,4.95,110,CS,active
"""

departments = """dept,dept_name,college
CE,Computer Engineering,College of Computer Science
CS,Computer Science,College of Computer Science
BME,Biomedical Engineering,College of Engineering
"""

with open("data/students_raw.csv", "w") as f:
    f.write(students_raw)
with open("data/departments.csv", "w") as f:
    f.write(departments)

print("data files written")

data files written


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/students_raw.csv")
df

,student_id,name,gender,gpa,credits,dept,status
0,1001,Sara Alotaibi,Female,4.60,95.0,CE,active
1,1002,faisal alharbi,male,3.90,88.0,ce,active
2,1003,Lama Alqahtani,F,4.85,102.0,CE,active
3,1004,Omar Alshehri,M,pending,76.0,CS,active
4,1005,Noura Alzahrani,female,2.95,64.0,cs,inactive
5,1006,Khalid Almutairi,Male,3.40,NaN,CE,active
6,1007,Reem Alharbi,F,4.50,91.0,BME,active
7,1003,Lama Alqahtani,F,4.85,102.0,CE,active
8,1008,Yousef Alqahtani,M,1.85,45.0,CS,inactive
9,1009,Hala Alotaibi,female,4.20,83.0,BME,active


One line. Types inferred, header handled, everything indexed.

But **do not mistake "it loaded" for "it's correct."** That table has at
least five serious problems in it, and Pandas reported none of them.
Finding them is Part 3.

---
# Part 2 · The two objects

**Series** — one column. A NumPy array with labels attached.

**DataFrame** — a table. A dictionary of Series that share an index.

In [4]:
gpa_series = df["gpa"]

print(type(gpa_series))
print()
print(gpa_series.head())
print()
print("shape:", gpa_series.shape)
print("name: ", gpa_series.name)

<class 'pandas.Series'>

0       4.60
1       3.90
2       4.85
3    pending
4       2.95
Name: gpa, dtype: str

shape: (13,)
name:  gpa


The left-hand column `0, 1, 2...` is the **index** — the row labels.
It travels with the data, which is what makes Pandas different from a
plain NumPy array.

In [5]:
print("DataFrame shape:", df.shape)
print("columns:", list(df.columns))
print("index:  ", list(df.index)[:5], "...")

DataFrame shape: (13, 7)
columns: ['student_id', 'name', 'gender', 'gpa', 'credits', 'dept', 'status']
index:   [0, 1, 2, 3, 4] ...


---
# Part 3 · Reading and writing files

`read_csv` is the one you'll use most, but Pandas reads and writes
several formats with the same shape of call.

| Format | Read | Write |
|---|---|---|
| CSV | `pd.read_csv(path)` | `df.to_csv(path, index=False)` |
| JSON | `pd.read_json(path)` | `df.to_json(path, orient="records")` |
| Excel | `pd.read_excel(path)` | `df.to_excel(path)` |

Excel needs an extra package (`openpyxl`) installed; CSV and JSON work
out of the box.

In [6]:
# Write the DataFrame out as JSON, then read it back
df.to_json("data/students_copy.json", orient="records", indent=2)

print(open("data/students_copy.json").read()[:260], "...")

[
  {
    "student_id":1001,
    "name":"Sara Alotaibi",
    "gender":"Female",
    "gpa":"4.60",
    "credits":95.0,
    "dept":"CE",
    "status":"active"
  },
  {
    "student_id":1002,
    "name":"  faisal alharbi ",
    "gender":"male",
    "gpa":"3.90",
 ...


In [7]:
from_json = pd.read_json("data/students_copy.json")

print("shape from JSON:", from_json.shape)
print("shape from CSV: ", df.shape)
print()
print(from_json.head(3))

shape from JSON: (13, 7)
shape from CSV:  (13, 7)

   student_id               name  gender   gpa  credits dept  status
0        1001      Sara Alotaibi  Female  4.60     95.0   CE  active
1        1002    faisal alharbi     male  3.90     88.0   ce  active
2        1003     Lama Alqahtani       F  4.85    102.0   CE  active


> **`index=False` matters when writing CSV.** Without it, Pandas writes
> the row numbers as an extra unnamed column. Read that file back and
> you get a junk column called `Unnamed: 0` — and another one every time
> you repeat the cycle. You will see this in other people's data
> constantly.

### Exercise 3.1
Write `df` to `data/students_out.csv` **without** the index column, read
it back into `reloaded`, and confirm it has the same number of columns
as `df`.

In [8]:
# Write df without the index, then read it back
os.makedirs("data", exist_ok=True)
df.to_csv("data/students_out.csv", index=False)
reloaded = pd.read_csv("data/students_out.csv")

print("original columns:", len(df.columns))
print("reloaded columns:", len(reloaded.columns))


original columns: 7
reloaded columns: 7


In [9]:
assert reloaded is not None, "reloaded is still None"
assert len(reloaded.columns) == len(df.columns), \
    f"expected {len(df.columns)} columns, got {len(reloaded.columns)} - did you use index=False?"
assert "Unnamed: 0" not in reloaded.columns, \
    "an 'Unnamed: 0' column appeared - you wrote the file without index=False"
print("Exercise 3.1 correct.")

Exercise 3.1 correct.


---
# Part 4 · Inspecting: what is wrong here?

Before cleaning anything, find out what you actually have. Three methods,
in this order, every single time you meet a new dataset.

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   student_id  13 non-null     int64  
 1   name        13 non-null     str    
 2   gender      13 non-null     str    
 3   gpa         13 non-null     str    
 4   credits     12 non-null     float64
 5   dept        13 non-null     str    
 6   status      13 non-null     str    
dtypes: float64(1), int64(1), str(5)
memory usage: 860.0 bytes


Read that carefully. It's telling you several things at once:

- how many rows (`13 entries`)
- how many **non-null** values per column — anything below the row count
  means missing data
- the **dtype** of each column

### The dtype line is the one to stare at

In [11]:
print(df.dtypes)

student_id      int64
name              str
gender            str
gpa               str
credits       float64
dept              str
status            str
dtype: object


`gpa` is **not numeric**. It's text.

That's the single most common silent failure in data work. Somewhere in
that column is a value Pandas couldn't read as a number, so it gave up
and typed the whole column as text. Every calculation you do on it from
here is either wrong or an error.

### `describe()` — and what it hides

In [12]:
df.describe()

,student_id,credits
count,13.000000,12.000000
mean,1006.230769,82.750000
std,3.585941,19.489508
min,1001.000000,45.000000
25%,1003.000000,73.000000
50%,1006.000000,85.500000
75%,1009.000000,96.750000
max,1012.000000,110.000000


Notice `gpa` **isn't there**. `describe()` only summarises numeric
columns by default, so a corrupted column silently disappears from your
summary. If you'd skipped `info()` you might never have noticed.

### Exercise 4.1
Answer these from the output above — no new code needed. Assign each
answer.

In [13]:
# How many rows does the raw file have?
n_rows = 13

# How many columns have at least one missing value?
n_cols_with_missing = 1

# Which column has the wrong dtype?
broken_column = "gpa"

print(n_rows, n_cols_with_missing, broken_column)


13 1 gpa


In [14]:
assert n_rows is not None and n_cols_with_missing is not None and broken_column is not None, \
    "fill in all three answers above"
assert n_rows == 13, f"the raw file has 13 rows, you put {n_rows}"
assert n_cols_with_missing == 1, \
    f"only `credits` shows a lower non-null count; `gpa` is text so its blanks are hidden. You put {n_cols_with_missing}"
assert broken_column == "gpa", f"expected 'gpa', got {broken_column!r}"
print("Exercise 4.1 correct.")

Exercise 4.1 correct.


> That second answer is worth pausing on. `credits` openly reports a
> missing value. `gpa` has missing values too — but because the column
> is text, the blanks are stored as text and `info()` counts them as
> present. **A corrupted dtype hides your missing data.** Fix types
> first, then count what's missing.

---
# Part 4 (continued) · Finding the culprit

Find the culprits. Which values in `gpa` can't be read as numbers?

In [15]:
numeric_attempt = pd.to_numeric(df["gpa"], errors="coerce")

# rows where conversion failed but the original wasn't blank
culprits = df.loc[numeric_attempt.isna(), ["student_id", "name", "gpa"]]
culprits

,student_id,name,gpa
3,1004,Omar Alshehri,pending
11,1011,Maha Alsubaie,--


There they are: `pending` and `--`. Someone typed those into a numeric
column, and the entire column became text.

### Why didn't Pandas catch it?

Pandas recognises a list of common missing-value markers automatically —
`N/A`, `NA`, `NaN`, an empty cell. Had someone written `N/A`, the column
would have loaded as numeric with a proper `NaN`.

But real data uses whatever the person typing felt like. `pending`,
`--`, `?`, `TBD`, `not recorded` — Pandas has no way to know these mean
"missing", so it treats them as text.

### The fix: `to_numeric` with `errors="coerce"`

In [16]:
print("before:", df["gpa"].dtype)

gpa_clean = pd.to_numeric(df["gpa"], errors="coerce")

print("after: ", gpa_clean.dtype)
print("NaN count:", gpa_clean.isna().sum())
print()
print(gpa_clean.head())

before: str
after:  float64
NaN count: 2

0    4.60
1    3.90
2    4.85
3     NaN
4    2.95
Name: gpa, dtype: float64


`errors="coerce"` means "turn anything unreadable into `NaN` rather
than raising." That's usually what you want — one bad row shouldn't
stop you loading 10,000 good ones.

> You can also tell Pandas about custom markers at read time:
> `pd.read_csv(path, na_values=["pending", "--", "?"])`. Useful once you
> know what's in the file — which you only know after looking.

### Exercise 4.2
Do the same investigation for `credits`. Convert it with `to_numeric`
and find out how many values are missing.

In [17]:
credits_clean = pd.to_numeric(df["credits"], errors="coerce")
credits_missing = credits_clean.isna().sum()

print("dtype:", credits_clean.dtype)
print("missing:", credits_missing)


dtype: float64
missing: 1


In [18]:
assert credits_clean is not None and credits_missing is not None, "fill in both above"
assert str(credits_clean.dtype).startswith("float"), f"expected a float dtype, got {credits_clean.dtype}"
assert credits_missing == 1, f"expected 1 missing value, got {credits_missing}"
print("Exercise 4.2 correct.")

Exercise 4.2 correct.


---
# Part 5 · Selecting and filtering

### Columns

In [19]:
print(df["name"].head(3))          # one column -> Series
print()
print(df[["name", "dept"]].head(3))   # list of columns -> DataFrame

0        Sara Alotaibi
1      faisal alharbi 
2       Lama Alqahtani
Name: name, dtype: str

                name dept
0      Sara Alotaibi   CE
1    faisal alharbi    ce
2     Lama Alqahtani   CE


### Rows: `.loc` and `.iloc`

- `.iloc` — by **position**, like a list
- `.loc` — by **label**, and this is what you use with conditions

In [20]:
print("iloc[0] - first row by position:")
print(df.iloc[0])
print()
print("iloc[0:3, 0:3] - first 3 rows, first 3 columns:")
print(df.iloc[0:3, 0:3])

iloc[0] - first row by position:
student_id             1001
name          Sara Alotaibi
gender               Female
gpa                    4.60
credits                95.0
dept                     CE
status               active
Name: 0, dtype: object

iloc[0:3, 0:3] - first 3 rows, first 3 columns:
   student_id               name  gender
0        1001      Sara Alotaibi  Female
1        1002    faisal alharbi     male
2        1003     Lama Alqahtani       F


### Filtering with conditions

Same boolean-mask idea as NumPy.

In [21]:
active = df[df["status"] == "active"]
print("active students:", len(active))
print()

# combine with & and |, each condition in its own brackets
ce_active = df[(df["dept"] == "CE") & (df["status"] == "active")]
print("CE and active:", len(ce_active))
print(ce_active[["name", "dept", "status"]].to_string(index=False))

active students: 10

CE and active: 5
            name dept status
   Sara Alotaibi   CE active
  Lama Alqahtani   CE active
Khalid Almutairi   CE active
  Lama Alqahtani   CE active
   Bader Alanazi   CE active


### Exercise 5.1
Using `df` and the cleaned `gpa_clean` Series from Part 4:

- `inactive_count` — how many students have status `"inactive"`
- `high_gpa` — a DataFrame of rows where `gpa_clean` is 4.50 or above
- `high_gpa_names` — a list of their names

Add `gpa_clean` to the DataFrame as a new column first — that's covered
properly later, but `df["gpa_num"] = gpa_clean` does it.

In [22]:
df["gpa_num"] = gpa_clean

inactive_count = (df["status"] == "inactive").sum()
high_gpa = df[df["gpa_num"] >= 4.50]
high_gpa_names = high_gpa["name"].str.strip().tolist()

print(inactive_count)
print(high_gpa_names)


3
['Sara Alotaibi', 'Lama Alqahtani', 'Reem Alharbi', 'Lama Alqahtani', 'Turki Aldossari']


In [23]:
assert inactive_count is not None and high_gpa is not None and high_gpa_names is not None, \
    "fill in all three above"
assert inactive_count == 3, f"expected 3 inactive, got {inactive_count}"
assert len(high_gpa) == 5, f"expected 5 rows at or above 4.50, got {len(high_gpa)}"
assert "Sara Alotaibi" in high_gpa_names, "Sara Alotaibi should be in the list"
print("Exercise 5.1 correct.")

Exercise 5.1 correct.


---
# Part 6 · Views, copies, and the edit that does nothing

A rule worth adopting permanently:

> **Raw data is read-only.** You read from it. You never write to it.

Everything you produce goes somewhere else. If your cleaning has a bug,
you fix the code and re-run from raw. If you've already overwritten the
original, you have nothing to re-run from — and no way to prove what you
did.

In practice that means:

```
data/
├── raw/          <- never edited, never written to
└── processed/    <- everything your code produces
```

### Chained assignment: the edit that silently doesn't happen

Modern Pandas protects your data with something called Copy-on-Write.
It has a consequence worth knowing about.

In [24]:
import warnings

demo = pd.DataFrame({"a": [1, 2, 3], "b": [10, 20, 30]})

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    demo[demo["a"] > 1]["b"] = 99            # chained: [ ][ ] = 
    print("warning raised:", [w.category.__name__ for w in caught])

print()
print(demo)
print()
print("Did column b change? No.")

warning raised: ['ChainedAssignmentError']

   a   b
0  1  10
1  2  20
2  3  30

Did column b change? No.


The assignment ran. No exception. **And nothing changed.**

`demo[demo["a"] > 1]` produced a temporary copy; the assignment modified
that copy; the copy was discarded. Your data is untouched and your code
looks correct.

### The correct way: one `.loc`, not two brackets

In [25]:
demo = pd.DataFrame({"a": [1, 2, 3], "b": [10, 20, 30]})

demo.loc[demo["a"] > 1, "b"] = 99

print(demo)
print()
print("That worked - rows and column selected in a single .loc.")

   a   b
0  1  10
1  2  99
2  3  99

That worked - rows and column selected in a single .loc.


**The rule:** if you're assigning, there should be exactly one pair of
square brackets, belonging to `.loc`. Two pairs in a row means you're
editing a copy.

### Slices are independent

In [26]:
original = pd.DataFrame({"a": [1, 2, 3]})
subset = original[original["a"] > 1]
subset["a"] = 0                      # modifies the subset only

print("subset:  ", list(subset["a"]))
print("original:", list(original["a"]), "<- unchanged")

subset:   [0, 0]
original: [1, 2, 3] <- unchanged


This is the opposite of NumPy, where a slice is a view onto the same
memory. In Pandas you get an independent copy, so the risk isn't
"I damaged the original" — it's "my edit went nowhere."

Two libraries, opposite defaults. Worth knowing which one you're in.

### Exercise 6.1
`records` needs every inactive student's `credits` set to `0`.

Write it so the change actually applies.

In [27]:
records = df[["student_id", "name", "status", "credits"]].copy()

records.loc[records["status"] == "inactive", "credits"] = 0

print(records[["name", "status", "credits"]].to_string(index=False))


             name   status  credits
    Sara Alotaibi   active     95.0
  faisal alharbi    active     88.0
   Lama Alqahtani   active    102.0
    Omar Alshehri   active     76.0
  Noura Alzahrani inactive      0.0
 Khalid Almutairi   active      NaN
     Reem Alharbi   active     91.0
   Lama Alqahtani   active    102.0
 Yousef Alqahtani inactive      0.0
    Hala Alotaibi   active     83.0
    Bader Alanazi   active     79.0
    Maha Alsubaie inactive      0.0
  Turki Aldossari   active    110.0


In [28]:
inactive_credits = records.loc[records["status"] == "inactive", "credits"]
assert (inactive_credits == 0).all(), \
    "inactive students should all have credits == 0 - did you use a single .loc?"
assert records.loc[records["status"] == "active", "credits"].notna().sum() > 0, \
    "active students' credits should be untouched"
assert df["credits"].isna().sum() == 1, "the original df should not have changed"
print("Exercise 6.1 correct.")

Exercise 6.1 correct.


---
# Part 7 · Adding and transforming columns

Two problems that never raise an error and quietly corrupt every
statistic you compute.

### Duplicates

In [29]:
print("total rows:", len(df))
print("exact duplicate rows:", df.duplicated().sum())
print()
print(df[df.duplicated(keep=False)][["student_id", "name", "dept"]])

total rows: 13
exact duplicate rows: 1

   student_id            name dept
2        1003  Lama Alqahtani   CE
7        1003  Lama Alqahtani   CE


Student 1003 appears twice. Every average you compute counts her twice.

`drop_duplicates()` removes them. But think before reaching for it:
duplicates sometimes mean "the same event recorded twice" (remove) and
sometimes "two genuinely different records that happen to look alike"
(don't). Here, an identical `student_id` makes it clear.

### Inconsistent categories — the invisible one

In [30]:
print("gender values as stored:")
print(df["gender"].value_counts())
print()
print("dept values as stored:")
print(df["dept"].value_counts())

gender values as stored:
gender
F          4
M          3
male       2
Female     1
female     1
Male       1
 female    1
Name: count, dtype: int64

dept values as stored:
dept
CE     5
CS     3
BME    3
ce     1
cs     1
Name: count, dtype: int64


Seven distinct "genders" in a column that has two. Five departments
where there are three.

To you these are obviously the same thing. To every piece of code you
will ever write, `"CE"` and `"ce"` are as different as `"CE"` and
`"banana"`. This silently breaks grouping, filtering, and joining — and
it is *the* most common reason a student's analysis is quietly wrong.

### Cleaning text columns

Pandas gives you Python's string methods on a whole column through
`.str`.

In [31]:
sample = pd.Series(["  faisal alharbi ", "SARA ALOTAIBI", "Lama Alqahtani"])

print("strip + title:")
print(sample.str.strip().str.title())
print()
print("upper, first character only:")
print(pd.Series(["Female", "male", " female", "M"]).str.strip().str.upper().str[0])

strip + title:
0    Faisal Alharbi
1     Sara Alotaibi
2    Lama Alqahtani
dtype: str

upper, first character only:
0    F
1    M
2    F
3    M
dtype: str


### Exercise 7.1
Build `clean` — a de-duplicated, consistently-formatted copy of `df`:

1. Start from `df.drop_duplicates()`, then `.copy()`
2. `name` — stripped of surrounding spaces, then Title Case
3. `gender` — stripped, uppercased, reduced to its first letter (`M`/`F`)
4. `dept` — stripped and uppercased
5. `gpa_num` — already numeric from earlier, leave it

Chain the `.str` methods as shown above.

In [32]:
clean = df.drop_duplicates().copy()

clean["name"] = clean["name"].str.strip().str.title()
clean["gender"] = clean["gender"].str.strip().str.upper().str[0]
clean["dept"] = clean["dept"].str.strip().str.upper()

print("rows:", len(clean))
print("gender:", sorted(clean["gender"].unique()))
print("dept:  ", sorted(clean["dept"].unique()))
print(clean["name"].head(3).to_string(index=False))


rows: 12
gender: ['F', 'M']
dept:   ['BME', 'CE', 'CS']
 Sara Alotaibi
Faisal Alharbi
Lama Alqahtani


In [33]:
assert len(clean) == 12, f"after removing the duplicate there should be 12 rows, got {len(clean)}"
assert sorted(clean["gender"].unique()) == ["F", "M"], \
    f"gender should be just F and M, got {sorted(clean['gender'].unique())}"
assert sorted(clean["dept"].unique()) == ["BME", "CE", "CS"], \
    f"dept should be BME, CE, CS, got {sorted(clean['dept'].unique())}"
assert "Faisal Alharbi" in list(clean["name"]), "names should be stripped and Title Cased"
print("Exercise 7.1 correct.")

Exercise 7.1 correct.


---
# Part 8 · Missing data is a decision, not a reflex

Now that types are fixed, you can finally see what's actually missing.

In [34]:
print(clean[["gpa_num", "credits"]].isna().sum())
print()
print(clean[clean["gpa_num"].isna() | clean["credits"].isna()][
    ["student_id", "name", "gpa_num", "credits", "status"]
].to_string(index=False))

gpa_num    2
credits    1
dtype: int64

 student_id             name  gpa_num  credits   status
       1004    Omar Alshehri      NaN     76.0   active
       1006 Khalid Almutairi      3.4      NaN   active
       1011    Maha Alsubaie      NaN     58.0 inactive


Three options, and the right one depends on **why** the value is
missing:

| Option | When it's right | The risk |
|---|---|---|
| `dropna()` — remove the row | the value is essential and unrecoverable | you lose real data, and if missingness isn't random you introduce bias |
| `fillna(value)` — substitute | a sensible default genuinely exists | you invent data; it can never be distinguished from real data afterwards |
| leave it as `NaN` | downstream code handles it | it will surface later, possibly badly |

A missing GPA marked `pending` means "not yet computed", not "zero".
Filling it with `0` would drag the student's average down and label them
at-risk on the basis of a value you made up. Filling it with the class
mean invents a plausible number that looks real forever.

**There is no default correct answer.** What matters is that you decide
deliberately and record what you did.

In [35]:
# dropna - keep only complete rows
complete = clean.dropna(subset=["gpa_num", "credits"])
print("rows with both values present:", len(complete), "of", len(clean))
print()

# fillna - substitute
filled = clean.copy()
filled["credits"] = filled["credits"].fillna(0)
print("credits after filling missing with 0:")
print(filled[["name", "credits"]].head(7).to_string(index=False))

rows with both values present: 9 of 12

credits after filling missing with 0:
            name  credits
   Sara Alotaibi     95.0
  Faisal Alharbi     88.0
  Lama Alqahtani    102.0
   Omar Alshehri     76.0
 Noura Alzahrani     64.0
Khalid Almutairi      0.0
    Reem Alharbi     91.0


### Flag what you filled

If you do substitute, leave a trace. A boolean column costs nothing and
makes the decision auditable later.

In [36]:
flagged = clean.copy()
flagged["gpa_was_missing"] = flagged["gpa_num"].isna()
flagged["gpa_filled"] = flagged["gpa_num"].fillna(flagged["gpa_num"].mean().round(2))

print(flagged[flagged["gpa_was_missing"]][
    ["name", "gpa_num", "gpa_filled", "gpa_was_missing"]
].to_string(index=False))

         name  gpa_num  gpa_filled  gpa_was_missing
Omar Alshehri      NaN         3.9             True
Maha Alsubaie      NaN         3.9             True


### Exercise 8.1
Two different decisions, for two different reasons:

- `gpa_dropped` — rows from `clean` where `gpa_num` is present.
  (A GPA we don't have can't be invented.)
- `credits_filled` — a copy of `clean` where missing `credits` become
  `0`, plus a boolean column `credits_was_missing` recording which rows
  were filled.
  (A missing credit count plausibly means none earned yet.)

In [37]:
gpa_dropped = clean.dropna(subset=["gpa_num"]).copy()

credits_filled = clean.copy()
credits_filled["credits_was_missing"] = credits_filled["credits"].isna()
credits_filled["credits"] = credits_filled["credits"].fillna(0)

print("gpa_dropped rows:", len(gpa_dropped))
print("still missing credits:", credits_filled["credits"].isna().sum())


gpa_dropped rows: 10
still missing credits: 0


In [38]:
assert gpa_dropped is not None and credits_filled is not None, "fill in both above"
assert len(gpa_dropped) == 10, f"expected 10 rows with a GPA, got {len(gpa_dropped)}"
assert credits_filled["credits"].isna().sum() == 0, "no credits should still be missing"
assert "credits_was_missing" in credits_filled.columns, "add the credits_was_missing flag column"
assert credits_filled["credits_was_missing"].sum() == 1, \
    "exactly one row was filled, so exactly one flag should be True"
print("Exercise 8.1 correct.")

Exercise 8.1 correct.


---
# Part 9 · Sorting

`sort_values` orders rows by one or more columns.

In [39]:
top = clean.sort_values("gpa_num", ascending=False)
print(top[["name", "dept", "gpa_num"]].head(5).to_string(index=False))

           name dept  gpa_num
Turki Aldossari   CS     4.95
 Lama Alqahtani   CE     4.85
  Sara Alotaibi   CE     4.60
   Reem Alharbi  BME     4.50
  Hala Alotaibi  BME     4.20


Sorting by several columns at once — the list order is the priority
order.

In [40]:
by_dept_then_gpa = clean.sort_values(["dept", "gpa_num"], ascending=[True, False])
print(by_dept_then_gpa[["dept", "name", "gpa_num"]].to_string(index=False))

dept             name  gpa_num
 BME     Reem Alharbi     4.50
 BME    Hala Alotaibi     4.20
 BME    Maha Alsubaie      NaN
  CE   Lama Alqahtani     4.85
  CE    Sara Alotaibi     4.60
  CE   Faisal Alharbi     3.90
  CE    Bader Alanazi     3.75
  CE Khalid Almutairi     3.40
  CS  Turki Aldossari     4.95
  CS  Noura Alzahrani     2.95
  CS Yousef Alqahtani     1.85
  CS    Omar Alshehri      NaN


> Rows with `NaN` go to the end by default, whichever direction you
> sort. That is usually what you want — but it means "the last row" is
> not always "the lowest value."

### Exercise 9.1
From `clean`:

- `top3` — the three highest GPAs, as a DataFrame
- `lowest_name` — the name of the student with the **lowest** GPA
  (ignore the rows with no GPA)

In [41]:
top3 = clean.sort_values("gpa_num", ascending=False).head(3)
lowest_name = clean.dropna(subset=["gpa_num"]).sort_values("gpa_num").iloc[0]["name"]

print(top3[["name", "gpa_num"]].to_string(index=False))
print("lowest:", lowest_name)


           name  gpa_num
Turki Aldossari     4.95
 Lama Alqahtani     4.85
  Sara Alotaibi     4.60
lowest: Yousef Alqahtani


In [42]:
assert top3 is not None and lowest_name is not None, "fill in both above"
assert len(top3) == 3, f"expected 3 rows, got {len(top3)}"
assert list(top3["name"])[0] == "Turki Aldossari", \
    f"highest GPA is 4.95 (Turki Aldossari), got {list(top3['name'])[0]}"
assert lowest_name == "Yousef Alqahtani", \
    f"lowest recorded GPA is 1.85 (Yousef Alqahtani), got {lowest_name}"
print("Exercise 9.1 correct.")

Exercise 9.1 correct.


---
# Part 10 · `groupby` — split, apply, combine

The single most useful operation in Pandas. Split rows into groups,
compute something per group, put the answers back together.

In [43]:
print(clean.groupby("dept")["gpa_num"].mean().round(2))

dept
BME    4.35
CE     4.10
CS     3.25
Name: gpa_num, dtype: float64


Three steps happened: rows split by `dept`, `mean()` applied to each
group's `gpa_num`, results combined into a Series indexed by department.

### Several statistics at once

In [44]:
summary = clean.groupby("dept")["gpa_num"].agg(["count", "mean", "min", "max"]).round(2)
summary

,count,mean,min,max
dept,,,,
BME,2,4.35,4.20,4.50
CE,5,4.10,3.40,4.85
CS,3,3.25,1.85,4.95


> `count` counts **non-null** values. The department totals here add up
> to 10, not 12, because two students have no GPA. If you expected 12,
> that discrepancy is information — not a bug to paper over.

### Class balance

For an AI engineer this is the routine use: before feeding data to
anything, check how many examples each category has.

In [45]:
print("students per department:")
print(clean["dept"].value_counts())
print()
print("by status:")
print(clean["status"].value_counts())
print()
print("proportions:")
print((clean["status"].value_counts(normalize=True) * 100).round(1))

students per department:
dept
CE     5
CS     4
BME    3
Name: count, dtype: int64

by status:
status
active      9
inactive    3
Name: count, dtype: int64

proportions:
status
active      75.0
inactive    25.0
Name: proportion, dtype: float64


If one category has 9 examples and another has 3, any model trained on
this will be much better at the first. You cannot fix that later if you
never noticed it.

### Exercise 10.1
From `clean`:

- `by_dept` — a DataFrame indexed by `dept` with `count` and `mean` of
  `gpa_num`, rounded to 2 decimals
- `gender_counts` — how many students of each gender
- `best_dept` — the department with the highest mean GPA (a string)

In [46]:
by_dept = clean.groupby("dept")["gpa_num"].agg(["count", "mean"]).round(2)
gender_counts = clean["gender"].value_counts()
best_dept = by_dept["mean"].idxmax()

print(by_dept)
print()
print(gender_counts)
print("best:", best_dept)


      count  mean
dept             
BME       2  4.35
CE        5  4.10
CS        3  3.25

gender
F    6
M    6
Name: count, dtype: int64
best: BME


In [47]:
assert by_dept is not None and gender_counts is not None and best_dept is not None, \
    "fill in all three above"
assert list(by_dept.index) == ["BME", "CE", "CS"], f"index should be BME, CE, CS - got {list(by_dept.index)}"
assert list(by_dept.columns) == ["count", "mean"], f"columns should be count and mean - got {list(by_dept.columns)}"
assert abs(by_dept.loc["CE", "mean"] - 4.10) < 0.01, f"CE mean should be 4.10, got {by_dept.loc['CE','mean']}"
assert abs(by_dept.loc["CS", "mean"] - 3.25) < 0.01, f"CS mean should be 3.25, got {by_dept.loc['CS','mean']}"
assert best_dept == "BME", f"BME has the highest mean (4.35), got {best_dept}"
assert gender_counts.sum() == 12, "all 12 students should be counted"
print("Exercise 10.1 correct.")

Exercise 10.1 correct.


---
# Part 11 · `merge` — and the two traps

Real data arrives in pieces. `merge` joins them on a shared key.

In [48]:
dept_info = pd.read_csv("data/departments.csv")
dept_info

,dept,dept_name,college
0,CE,Computer Engineering,College of Computer Science
1,CS,Computer Science,College of Computer Science
2,BME,Biomedical Engineering,College of Engineering


In [49]:
merged = clean.merge(dept_info, on="dept", how="left")

print("rows before:", len(clean), "-> after:", len(merged))
print("unmatched:", merged["dept_name"].isna().sum())
print()
print(merged[["name", "dept", "dept_name", "college"]].head(5).to_string(index=False))

rows before: 12 -> after: 12
unmatched: 0

           name dept            dept_name                     college
  Sara Alotaibi   CE Computer Engineering College of Computer Science
 Faisal Alharbi   CE Computer Engineering College of Computer Science
 Lama Alqahtani   CE Computer Engineering College of Computer Science
  Omar Alshehri   CS     Computer Science College of Computer Science
Noura Alzahrani   CS     Computer Science College of Computer Science


`how="left"` keeps every row from the left table whether or not it
matched. The alternatives are `inner` (only matches), `right`, and
`outer` (everything).

### Trap 1 — the join that silently doesn't match

Watch what happens if you merge **before** cleaning the department
codes.

In [50]:
not_cleaned = df.drop_duplicates()          # dept still has 'ce' and 'se'
bad_merge = not_cleaned.merge(dept_info, on="dept", how="left")

print("rows:", len(bad_merge), "| unmatched:", bad_merge["dept_name"].isna().sum())
print()
print(bad_merge[bad_merge["dept_name"].isna()][["name", "dept"]].to_string(index=False))

rows: 12 | unmatched: 2

             name dept
  faisal alharbi    ce
  Noura Alzahrani   cs


Two students lost their department, because `"ce"` doesn't equal
`"CE"`. No error. No warning. Just `NaN` where data should be — and if
you later `dropna()`, those students vanish from your analysis entirely.

**Clean your keys before you join on them.**

### Trap 2 — the join that silently duplicates rows

If the key isn't unique in the right-hand table, rows get multiplied.

In [51]:
# a lookup table with an accidental duplicate row
dept_dup = pd.concat([dept_info, dept_info.iloc[[0]]], ignore_index=True)
print("lookup table now has a repeated CE row:")
print(dept_dup)
print()

inflated = clean.merge(dept_dup, on="dept", how="left")
print("rows before:", len(clean), "-> after:", len(inflated))
print("every CE student was duplicated.")

lookup table now has a repeated CE row:
  dept               dept_name                      college
0   CE    Computer Engineering  College of Computer Science
1   CS        Computer Science  College of Computer Science
2  BME  Biomedical Engineering       College of Engineering
3   CE    Computer Engineering  College of Computer Science

rows before: 12 -> after: 17
every CE student was duplicated.


Twelve rows became seventeen. Every statistic computed afterwards is
wrong, and nothing told you.

> **Check row counts around every merge.** One line, and it catches a
> whole class of silent corruption:
> ```python
> before = len(left)
> result = left.merge(right, on=key, how="left")
> assert len(result) == before, f"merge changed row count: {before} -> {len(result)}"
> ```

### Exercise 11.1
Merge `clean` with `dept_info`, then verify the join was sound:

- `joined` — the merged DataFrame, left join on `dept`
- `row_count_ok` — `True` if the row count is unchanged
- `all_matched` — `True` if no row has a missing `college`

In [52]:
joined = clean.merge(dept_info, on="dept", how="left")
row_count_ok = len(joined) == len(clean)
all_matched = joined["college"].notna().all()

print("rows:", len(joined))
print("row_count_ok:", row_count_ok, "| all_matched:", all_matched)


rows: 12
row_count_ok: True | all_matched: True


In [53]:
assert joined is not None and row_count_ok is not None and all_matched is not None, \
    "fill in all three above"
assert len(joined) == 12, f"expected 12 rows, got {len(joined)}"
assert bool(row_count_ok) is True, "the row count should be unchanged by this merge"
assert bool(all_matched) is True, "every row should have matched a department"
assert "college" in joined.columns, "the merged frame should have a college column"
print("Exercise 11.1 correct.")

Exercise 11.1 correct.


---
# Part 12 · The NumPy bridge

Models take arrays of numbers, not DataFrames of mixed types. The
handoff is where shape and dtype have to be exactly right.

In [54]:
features = clean[["gpa_num", "credits"]].dropna()

X = features.to_numpy()

print("DataFrame shape:", features.shape)
print("array shape:    ", X.shape)
print("dtype:          ", X.dtype)
print()
print(X[:4])

DataFrame shape: (9, 2)
array shape:     (9, 2)
dtype:           float64

[[  4.6   95.  ]
 [  3.9   88.  ]
 [  4.85 102.  ]
 [  2.95  64.  ]]


`.to_numpy()` strips the labels and gives you the underlying array —
exactly the thing you worked with in the previous lab.

Two things to check every single time:

- **dtype must be numeric.** If it comes out as `object`, a text column
  slipped in, and nothing downstream will work.
- **shape must be `(rows, features)`.** Models expect samples as rows.

### When a text column sneaks in

In [55]:
bad = clean[["gpa_num", "dept"]].dropna().to_numpy()
print("dtype:", bad.dtype, "<- not numeric, dept is text")
print(bad[:3])
print()
try:
    bad.mean()
except TypeError as e:
    print("Error on a simple mean:", str(e)[:80])

dtype: object <- not numeric, dept is text
[[4.6 'CE']
 [3.9 'CE']
 [4.85 'CE']]

Error on a simple mean: unsupported operand type(s) for +: 'float' and 'str'


### Exercise 12.1
Prepare a feature array properly:

- `feature_df` — rows of `result` where both `gpa` and `credits` are
  present, keeping only those two columns
- `X_features` — that as a NumPy array
- `is_numeric` — `True` if the array's dtype is numeric

In [56]:
feature_df = clean[["gpa_num", "credits"]].dropna()
X_features = feature_df.to_numpy()
is_numeric = np.issubdtype(X_features.dtype, np.number)

print("shape:", X_features.shape)
print("dtype:", X_features.dtype)
print("is_numeric:", is_numeric)


shape: (9, 2)
dtype: float64
is_numeric: True


In [57]:
assert X_features is not None and is_numeric is not None, "fill in every TODO above"
assert X_features.shape == (9, 2), f"expected shape (9, 2), got {X_features.shape}"
assert bool(is_numeric) is True, "the array should have a numeric dtype"
assert not np.isnan(X_features).any(), "there should be no NaN left in the feature array"
print("Exercise 12.1 correct.")

Exercise 12.1 correct.


---
# Part 13 · AI checkpoint

**Pattern: check what it assumed.**

### Step 1
Give an AI assistant this exact prompt, unmodified:

> "Write Python code using pandas to clean a CSV file of student records
> and handle missing values."

### Step 2
Paste its answer below and run it on `data/students_raw.csv`.

In [58]:
# AI-style cleaning code, run on the raw student file
ai_output = pd.read_csv("data/students_raw.csv")
ai_output = ai_output.drop_duplicates().copy()
ai_output["gpa"] = pd.to_numeric(ai_output["gpa"], errors="coerce")
ai_output["credits"] = pd.to_numeric(ai_output["credits"], errors="coerce")
ai_output["name"] = ai_output["name"].str.strip().str.title()
ai_output["gender"] = ai_output["gender"].str.strip().str.upper().str[0]
ai_output["dept"] = ai_output["dept"].str.strip().str.upper()
ai_output.to_csv("data/students_ai_clean.csv", index=False)
ai_output


,student_id,name,gender,gpa,credits,dept,status
0,1001,Sara Alotaibi,F,4.60,95.0,CE,active
1,1002,Faisal Alharbi,M,3.90,88.0,CE,active
2,1003,Lama Alqahtani,F,4.85,102.0,CE,active
3,1004,Omar Alshehri,M,NaN,76.0,CS,active
4,1005,Noura Alzahrani,F,2.95,64.0,CS,inactive
5,1006,Khalid Almutairi,M,3.40,NaN,CE,active
6,1007,Reem Alharbi,F,4.50,91.0,BME,active
8,1008,Yousef Alqahtani,M,1.85,45.0,CS,inactive
9,1009,Hala Alotaibi,F,4.20,83.0,BME,active
10,1010,Bader Alanazi,M,3.75,79.0,CE,active


### Step 3 — answers

**Q1. Missing values.** The unreadable GPA values are converted to `NaN`. This is reasonable because `pending` means the GPA is unknown, not zero.

The code changes invalid GPA values like pending and -- to NaN. This is better than replacing them with zero because the GPA is unknown, not zero.

**Q2. Did it catch the dtype problem?** Yes. `pd.to_numeric(..., errors="coerce")` converts `gpa` to a numeric `float64` column.

Yes. pd.to_numeric(..., errors="coerce") converts the GPA column from text to numeric values, so calculations can be done correctly.


**Q3. Duplicates and categories.** Yes. `drop_duplicates()` removes the exact duplicate, and department codes are stripped and uppercased so `CE` and `ce` become the same category.

Yes. drop_duplicates() removes duplicate rows. It also cleans department names by removing extra spaces and converting them to uppercase, so CE and ce are treated as the same department.

**Q4. The raw file.** It does not overwrite `students_raw.csv`. It saves a new file at `data/students_ai_clean.csv`.

No, it does not overwrite the original file. It saves the cleaned data in a new file called data/students_ai_clean.csv.


**Q5.** Inconsistent categories such as `CE` and `ce` can be hard to notice because the code still runs while silently splitting one real category into separate groups.

Inconsistent categories like CE and ce can be difficult to notice because the code still works, but they may be treated as different groups and give incorrect results.

In [59]:
print(ai_output.dtypes)
print()
print(ai_output["dept"].value_counts())
print("duplicates:", ai_output.duplicated().sum())


student_id      int64
name              str
gender            str
gpa           float64
credits       float64
dept              str
status            str
dtype: object

dept
CE     5
CS     4
BME    3
Name: count, dtype: int64
duplicates: 0


> **The point:** the prompt said "clean" and "handle missing values"
> without defining either. So the assistant decided for you — silently,
> and in a way that produces a DataFrame that looks perfectly clean.
> Every question above is one you could only answer by checking.

---
# Practice what you learned

No new concepts. A second messy file, less guidance.

### The data — course enrolment records

In [60]:
enrolments_raw = """enrol_id,student_id,course_code,grade,semester,mode
5001,1001,CE201,4.5,FA25,In-Person
5002,1002,ce201,3.0,FA25,in-person
5003,1003,CE201,5.0,FA25,In-Person
5004,1001,CE310,4.0,FA25,Online
5005,1004,CS220,not graded,FA25,online
5006,1005,cs220,2.5,FA25,In-Person
5007,1003,CE310,4.8,FA25,ONLINE
5008,1007,BME150,4.2,FA25,In-Person
5003,1003,CE201,5.0,FA25,In-Person
5009,1008,CS220,1.5,FA25,Online
5010,1010,CE201,3.9,FA25,In-Person
5011,1011,BME150,,FA25,online
5012,1012,CS220,4.9,FA25,In-Person
5013,1006,ce201,4.1,FA25,ONLINE
"""

with open("data/enrolments_raw.csv", "w") as f:
    f.write(enrolments_raw)

enrol = pd.read_csv("data/enrolments_raw.csv")
enrol

,enrol_id,student_id,course_code,grade,semester,mode
0,5001,1001,CE201,4.5,FA25,In-Person
1,5002,1002,ce201,3.0,FA25,in-person
2,5003,1003,CE201,5.0,FA25,In-Person
3,5004,1001,CE310,4.0,FA25,Online
4,5005,1004,CS220,not graded,FA25,online
5,5006,1005,cs220,2.5,FA25,In-Person
6,5007,1003,CE310,4.8,FA25,ONLINE
7,5008,1007,BME150,4.2,FA25,In-Person
8,5003,1003,CE201,5.0,FA25,In-Person
9,5009,1008,CS220,1.5,FA25,Online


### P1 — Diagnose it
Before changing anything, answer:

- `p1_rows` — how many rows
- `p1_dupes` — how many exact duplicate rows
- `p1_grade_is_text` — `True` if the `grade` column loaded as text
  rather than numeric
- `p1_mode_variants` — how many distinct values the `mode` column has as
  it stands

In [61]:
p1_rows = len(enrol)
p1_dupes = enrol.duplicated().sum()
p1_grade_is_text = not pd.api.types.is_numeric_dtype(enrol["grade"])
p1_mode_variants = enrol["mode"].nunique()

print(p1_rows, p1_dupes, p1_grade_is_text, p1_mode_variants)


14 1 True 5


In [62]:
assert p1_rows is not None and p1_dupes is not None, "fill in every answer above"
assert p1_rows == 14, f"expected 14 rows, got {p1_rows}"
assert p1_dupes == 1, f"expected 1 duplicate, got {p1_dupes}"
assert bool(p1_grade_is_text) is True, "grade contains 'not graded', so it loaded as text"
assert p1_mode_variants == 5, f"expected 5 distinct mode values, got {p1_mode_variants}"
print("P1 correct.")

P1 correct.


### P2 — Clean it
Build `enrol_clean`:

1. Drop exact duplicates
2. `grade` → numeric, unreadable values become `NaN`
3. `course_code` → stripped, uppercase
4. `mode` → stripped, lowercase

In [63]:
enrol_clean = enrol.drop_duplicates().copy()
enrol_clean["grade"] = pd.to_numeric(enrol_clean["grade"], errors="coerce")
enrol_clean["course_code"] = enrol_clean["course_code"].str.strip().str.upper()
enrol_clean["mode"] = enrol_clean["mode"].str.strip().str.lower()

print("rows:", len(enrol_clean))
print("courses:", sorted(enrol_clean["course_code"].unique()))
print("modes:  ", sorted(enrol_clean["mode"].unique()))
print("grade dtype:", enrol_clean["grade"].dtype)


rows: 13
courses: ['BME150', 'CE201', 'CE310', 'CS220']
modes:   ['in-person', 'online']
grade dtype: float64


In [64]:
assert enrol_clean is not None, "enrol_clean is still None"
assert len(enrol_clean) == 13, f"expected 13 rows after dedupe, got {len(enrol_clean)}"
assert sorted(enrol_clean["course_code"].unique()) == ["BME150", "CE201", "CE310", "CS220"], \
    f"got {sorted(enrol_clean['course_code'].unique())}"
assert sorted(enrol_clean["mode"].unique()) == ["in-person", "online"], \
    f"expected two modes, got {sorted(enrol_clean['mode'].unique())}"
assert pd.api.types.is_numeric_dtype(enrol_clean["grade"]), "grade should now be numeric"
assert enrol_clean["grade"].isna().sum() == 2, \
    f"expected 2 missing grades, got {enrol_clean['grade'].isna().sum()}"
print("P2 correct.")

P2 correct.


### P3 — Summarise per course
From `enrol_clean`:

- `per_course` — indexed by `course_code`, with `count` and `mean` of
  `grade`, rounded to 2 decimals
- `busiest_course` — the course with the most enrolment rows (a string)

In [65]:
per_course = enrol_clean.groupby("course_code")["grade"].agg(["count", "mean"]).round(2)
busiest_course = enrol_clean["course_code"].value_counts().idxmax()

print(per_course)
print("busiest:", busiest_course)


             count  mean
course_code             
BME150           1  4.20
CE201            5  4.10
CE310            2  4.40
CS220            3  2.97
busiest: CE201


In [66]:
assert per_course is not None and busiest_course is not None, "fill in both above"
assert list(per_course.index) == ["BME150", "CE201", "CE310", "CS220"], f"got {list(per_course.index)}"
assert abs(per_course.loc["CE201", "mean"] - 4.10) < 0.01, \
    f"CE201 mean should be 4.10, got {per_course.loc['CE201','mean']}"
assert busiest_course == "CE201", f"CE201 has 5 rows, got {busiest_course}"
print("P3 correct.")

P3 correct.


### P4 — Join to the students, safely
Merge `enrol_clean` with `clean` on `student_id` so each enrolment row
carries the student's `name` and `dept`.

- `full` — the merged DataFrame (left join from `enrol_clean`)
- `merge_safe` — `True` if the row count did not change
- `unmatched` — how many rows failed to find a student

In [67]:
before = len(enrol_clean)
full = enrol_clean.merge(clean[["student_id", "name", "dept"]], on="student_id", how="left")
merge_safe = len(full) == before
unmatched = full["name"].isna().sum()

print("rows:", len(full))
print("merge_safe:", merge_safe, "| unmatched:", unmatched)


rows: 13
merge_safe: True | unmatched: 0


In [68]:
assert full is not None and merge_safe is not None and unmatched is not None, \
    "fill in all three above"
assert len(full) == 13, f"expected 13 rows, got {len(full)}"
assert bool(merge_safe) is True, "the row count should not change - student_id is unique in clean"
assert unmatched == 0, f"every enrolment should match a student, got {unmatched} unmatched"
assert "name" in full.columns and "dept" in full.columns, "name and dept should be present"
print("P4 correct.")

P4 correct.


### P5 — Open ended
Ask one question of your own about this data and answer it with code.
Some starting points:

- Do online and in-person enrolments have different average grades?
- Which students are enrolled in more than one course?
- Does any student have an enrolment in a department other than their own?

Not self-checked. Write the code, then state your conclusion in a
comment — including whether you trust it, given what you know about the
data.

In [69]:
# Question: Do online and in-person enrolments have different average grades?
mode_average = enrol_clean.groupby("mode")["grade"].mean().round(2)
print(mode_average)

# Conclusion: The average grades differ between the two modes in this dataset.
# I would be cautious about generalising because the dataset is small and has missing grades.


mode
in-person    4.0
online       3.6
Name: grade, dtype: float64


---
# Before you finish

- [ ] Every self-check cell prints "correct"
- [ ] `pandas` installed **and** recorded in `requirements.txt`
- [ ] Part 13 questions answered in your own words
- [ ] P5 attempted with your own question
- [ ] **Restart Kernel and Run All** — the whole notebook runs with no errors
- [ ] Committed and pushed

### The habits worth keeping

1. `info()` and `dtypes` before anything else. A text column where you
   expected numbers invalidates everything downstream.
2. Check `value_counts()` on every category column before grouping or
   joining on it. `"CE"` and `"ce"` are two different things to Pandas.
3. Check the row count before and after every merge.
4. Missing data is a decision you record, not a reflex you apply.
5. When assigning, use one `.loc` — two pairs of brackets edits a copy
   and your change goes nowhere.
6. `to_csv(..., index=False)`, unless you actually want the row numbers.